In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1560_Bawana_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,182.58,294.75,3.16,10.06,13.21,38.35,1.73,0.95,8.22,...,NaN,11.67,75.98,0.62,158.50,0.0,0.0,32.42,988.06,NaN
1,2024-01-02,186.58,292.04,4.81,11.92,16.74,31.06,8.63,1.00,8.85,...,NaN,11.45,73.46,0.74,157.69,0.0,0.0,49.65,987.95,NaN
2,2024-01-03,206.87,358.75,11.04,22.48,24.12,30.40,3.43,1.53,11.66,...,NaN,11.21,80.81,0.68,193.54,0.0,0.0,32.87,988.02,NaN
3,2024-01-04,195.38,309.02,9.29,24.05,20.35,27.83,4.30,1.35,4.68,...,NaN,10.77,86.05,0.57,138.94,0.0,0.0,15.66,988.00,NaN
4,2024-01-05,142.25,236.91,5.59,18.52,14.34,41.94,5.00,1.28,10.03,...,NaN,10.96,85.93,0.70,160.05,0.0,0.0,14.22,988.00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,183.30,222.57,3.65,26.42,17.14,50.28,8.84,0.92,20.75,...,NaN,15.50,85.47,1.30,236.74,0.0,0.0,8.28,999.00,NaN
362,2024-12-28,124.67,148.76,6.36,26.94,19.44,34.60,8.87,0.93,7.25,...,NaN,15.81,88.61,0.76,198.14,0.0,0.0,14.74,999.00,NaN
363,2024-12-29,90.00,109.30,1.23,13.28,7.98,27.18,6.92,0.87,21.80,...,NaN,15.08,84.48,1.04,152.26,0.0,0.0,42.41,998.90,NaN
364,2024-12-30,109.67,138.17,1.62,14.93,9.24,25.81,6.77,0.72,33.02,...,NaN,13.84,80.18,0.78,143.62,0.0,0.0,41.97,998.98,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         182.58        294.75        3.16        10.06   
1  2024-01-02         186.58        292.04        4.81        11.92   
2  2024-01-03         206.87        358.75       11.04        22.48   
3  2024-01-04         195.38        309.02        9.29        24.05   
4  2024-01-05         142.25        236.91        5.59        18.52   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      13.21        38.35         1.73        0.95           8.22   
1      16.74        31.06         8.63        1.00           8.85   
2      24.12        30.40         3.43        1.53          11.66   
3      20.35        27.83         4.30        1.35           4.68   
4      14.34        41.94         5.00        1.28          10.03   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0             1.74            19.35    11.67   75.98      0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,0.996132,0.537766,-0.872545,-1.357247,-0.717305,0.749454,-1.453918,-0.367988,-1.688743,-0.096784,-0.156492,-1.634388,0.946409,-1.633124,-0.760741,-0.457212,-0.460876,-1.726755,-0.388119
1,2024-01-02,1.052228,0.514729,-0.389014,-1.212189,-0.314007,0.004562,-0.252976,-0.234993,-1.656798,-0.096784,-0.156492,-1.660551,0.771185,-1.338987,-0.780336,-0.457212,-0.460876,-1.361875,-0.412908
2,2024-01-03,1.336777,1.081823,1.436680,-0.388635,0.529148,-0.062877,-1.158034,1.174749,-1.514313,2.749058,-0.156492,-1.689092,1.282253,-1.486056,0.086905,-0.457212,-0.460876,-1.717225,-0.397133
3,2024-01-04,1.175640,0.659074,0.923844,-0.266193,0.098430,-0.325480,-1.006611,0.695968,-1.868243,1.168035,-0.701576,-1.741418,1.646606,-1.755681,-1.233914,-0.457212,-0.460876,-2.081681,-0.401640
4,2024-01-05,0.430541,0.046075,-0.160436,-0.697468,-0.588204,1.116280,-0.884776,0.509776,-1.596964,0.804400,0.405412,-1.718822,1.638262,-1.437033,-0.723246,-0.457212,-0.460876,-2.112176,-0.401640
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,1.006229,-0.075828,-0.728951,-0.081361,-0.268308,1.968462,-0.216425,-0.447785,-1.053393,-0.041448,0.593477,-1.178916,1.606277,0.033653,1.131949,-0.457212,-0.460876,-2.237968,2.077325
362,2024-12-28,0.183998,-0.703278,0.065211,-0.040807,-0.005536,0.366279,-0.211204,-0.421186,-1.737928,-0.239076,0.305263,-1.142050,1.824610,-1.289964,0.198183,-0.457212,-0.460876,-2.101164,2.077325
363,2024-12-29,-0.302216,-1.038723,-1.438129,-1.106125,-1.314826,-0.391897,-0.550601,-0.580779,-1.000151,-0.776624,-1.338400,-1.228863,1.537439,-0.603644,-0.911692,-0.457212,-0.460876,-1.515197,2.054789
364,2024-12-30,-0.026363,-0.793303,-1.323840,-0.977445,-1.170872,-0.531883,-0.576708,-0.979763,-0.431227,-0.855675,-1.386563,-1.376326,1.238447,-1.240941,-1.120701,-0.457212,-0.460876,-1.524515,2.072818


In [10]:
df.to_excel('bawana2024.xlsx', index=False)